# BM25 RAG con noticias de la Universidad de Oviedo

Este notebook muestra un flujo simple de RAG con BM25 y una opción para responder con un modelo local de Hugging Face, sin necesidad de API key.

La idea es muy directa:

1. Cargar noticias.
2. Crear fragmentos recuperables.
3. Indexarlos con BM25.
4. Recuperar contexto para una pregunta.
5. Responder usando ese contexto.


In [1]:
# Instala las dependencias necesarias dentro del notebook.
# En Google Colab puedes ejecutar esta celda tal cual.
%pip install -q rank-bm25 pandas transformers torch sentencepiece accelerate


## 1. Cargar el dataset

Si ejecutas el notebook desde la raíz del repo, usará `data/json/noticias_uniovi.json`. Si lo ejecutas en Colab y el archivo no está en esa ruta, te pedirá subir el JSON.

In [2]:
import json
import math
import re
import textwrap
import unicodedata
from pathlib import Path

import pandas as pd
from rank_bm25 import BM25Okapi


DATA_PATH = Path("./noticias_uniovi.json")

with DATA_PATH.open("r", encoding="utf-8") as f:
    noticias = json.load(f)

df = pd.DataFrame(noticias)
print(f"Noticias cargadas: {len(df)}")
display(df[["fecha", "titulo", "etiquetas", "fuente_html"]].head(5))

Noticias cargadas: 303


,fecha,titulo,etiquetas,fuente_html
0,22/07/2026,La nueva Junta Rectora de CRUE se reúne con el...,[Información institucional],15310136.html
1,20/07/2026,La Universidad de Oviedo reconstruye los oríge...,[Investigación],15288681.html
2,20/07/2026,La Universidá Asturiana de Branu incorpora la ...,"[Cultura, Estudiantes]",15289547.html
3,17/07/2026,El grupo Llabor de la Universidad de Oviedo re...,"[Investigación, Cultura]",15271733.html
4,16/07/2026,"El 81,43% del estudiantado aprueba la PAU en l...","[Nota de prensa, Información institucional, Es...",15260773.html


## 2. Preparar texto y tokens

BM25 compara palabras clave de la pregunta con palabras clave de los fragmentos. Por eso normalizamos texto, quitamos acentos y eliminamos palabras muy comunes.


In [3]:
STOPWORDS = set("""
a al algo algunas algunos ante antes como con contra cual cuando de del desde donde durante e el ella ellas ellos en entre era
eran es esa esas ese eso esos esta estaba estaban estamos estan estar estas este esto estos fue fueron ha han hasta hay la las le
les lo los mas me mi mis mucha muchas mucho muchos muy no nos o para pero por que se si sin sobre son su sus tambien te tiene
tienen un una unas uno unos y ya universidad oviedo
""".split())


def strip_accents(text: str) -> str:
    text = unicodedata.normalize("NFKD", text)
    return "".join(ch for ch in text if not unicodedata.combining(ch))


def normalize_text(text: str) -> str:
    return strip_accents(str(text).lower())


def tokenize(text: str) -> list[str]:
    text = normalize_text(text)
    tokens = re.findall(r"[a-z0-9]+", text)
    return [tok for tok in tokens if len(tok) > 2 and tok not in STOPWORDS]


print(tokenize("¿Qué noticias hablan de inteligencia artificial y cursos de verano?"))

['noticias', 'hablan', 'inteligencia', 'artificial', 'cursos', 'verano']


## 3. Crear fragmentos recuperables

En lugar de trabajar con noticias completas, convertimos cada noticia en fragmentos cortos para que la recuperación sea más precisa.


In [4]:
def clean_value(value) -> str:
    if isinstance(value, list):
        return ", ".join(map(str, value))
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return ""
    return str(value)


def noticia_to_text(row: pd.Series) -> str:
    parts = [
        f"Título: {clean_value(row.get('titulo'))}",
        f"Fecha: {clean_value(row.get('fecha'))}",
        f"Etiquetas: {clean_value(row.get('etiquetas'))}",
        f"Resumen: {clean_value(row.get('resumen'))}",
        f"Noticia: {clean_value(row.get('noticia'))}",
    ]
    return "\n".join(part for part in parts if part.strip())


def chunk_text(text: str, chunk_size: int = 140, overlap: int = 35) -> list[str]:
    words = text.split()
    if len(words) <= chunk_size:
        return [text]

    chunks = []
    step = chunk_size - overlap
    for start in range(0, len(words), step):
        chunk = " ".join(words[start:start + chunk_size])
        if chunk:
            chunks.append(chunk)
        if start + chunk_size >= len(words):
            break
    return chunks


chunks = []
for doc_id, row in df.iterrows():
    full_text = noticia_to_text(row)
    for chunk_id, chunk in enumerate(chunk_text(full_text)):
        chunks.append({
            "doc_id": int(doc_id),
            "chunk_id": int(chunk_id),
            "titulo": clean_value(row.get("titulo")),
            "fecha": clean_value(row.get("fecha")),
            "etiquetas": clean_value(row.get("etiquetas")),
            "fuente_html": clean_value(row.get("fuente_html")),
            "text": chunk,
        })

print(f"Fragmentos creados: {len(chunks)}")
print(textwrap.shorten(chunks[0]["text"], width=700, placeholder=" ..."))


Fragmentos creados: 2221
Título: La nueva Junta Rectora de CRUE se reúne con el presidente del Gobierno en La Moncloa Fecha: 22/07/2026 Etiquetas: Información institucional Resumen: El encuentro ha permitido abordar cuestiones como la financiación pública de las universidades, el problema del alojamiento universitario, la inteligencia artificial y su impacto en las universidades y la necesidad de continuar con programas de apoyo a la transformación digital de las universidades | La reunión ha contado con la participación de Ignacio Villaverde, rector de la Universidad de Oviedo y presidente de la sectorial de Secretarías Generales en la nueva dirección de CRUE Noticia: La Junta Rectora de la Conferencia de ...


## 4. Construir el índice BM25

BM25 asigna una puntuación a cada fragmento según las palabras compartidas con la pregunta.


In [5]:
tokenized_corpus = [tokenize(chunk["text"]) for chunk in chunks]
bm25 = BM25Okapi(tokenized_corpus, k1=1.5, b=0.75)

print("Índice BM25 listo")
print(f"Fragmentos indexados: {len(tokenized_corpus)}")


Índice BM25 listo
Fragmentos indexados: 2221


## 5. Recuperar contexto para una pregunta

Esta función es el retriever. Dada una pregunta, devuelve los `top_k` fragmentos con mayor puntuación BM25.

In [6]:
def search_bm25(query: str, top_k: int = 5) -> list[dict]:
    query_tokens = tokenize(query)
    scores = bm25.get_scores(query_tokens)
    top_indexes = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_k]

    results = []
    for rank, idx in enumerate(top_indexes, start=1):
        item = dict(chunks[idx])
        item["rank"] = rank
        item["score"] = float(scores[idx])
        item["matched_tokens"] = sorted(set(query_tokens) & set(tokenized_corpus[idx]))
        results.append(item)
    return results


query = "¿Qué noticias hablan de inteligencia artificial en la Universidad de Oviedo?"
results = search_bm25(query, top_k=5)

display(pd.DataFrame([{
    "rank": r["rank"],
    "score": round(r["score"], 3),
    "fecha": r["fecha"],
    "titulo": r["titulo"],
    "matched_tokens": ", ".join(r["matched_tokens"]),
} for r in results]))

,rank,score,fecha,titulo,matched_tokens
0,1,9.541,23/03/2026,La mayoría de los universitarios fallan al dis...,noticias
1,2,8.889,08/07/2026,"La Universidad de Oviedo desarrolla DubbiOvi, ...","artificial, inteligencia"
2,3,8.712,08/07/2026,"La Universidad de Oviedo desarrolla DubbiOvi, ...","artificial, inteligencia"
3,4,8.320,23/03/2026,La mayoría de los universitarios fallan al dis...,noticias
4,5,8.162,10/07/2026,La Universidad de Oviedo reúne a especialistas...,"artificial, inteligencia"


## 6. Construir el contexto RAG

El retriever no responde: solo selecciona texto. El siguiente paso del RAG es empaquetar esos fragmentos en un contexto que pueda usar un generador.

In [7]:
def build_context(results: list[dict], max_chars: int = 4500) -> str:
    blocks = []
    used_chars = 0

    for r in results:
        block = (
            f"[Fuente {r['rank']}]\n"
            f"Título: {r['titulo']}\n"
            f"Fecha: {r['fecha']}\n"
            f"Etiquetas: {r['etiquetas']}\n"
            f"Archivo: {r['fuente_html']}\n"
            f"Texto: {r['text']}"
        )
        if used_chars + len(block) > max_chars:
            break
        blocks.append(block)
        used_chars += len(block)

    return "\n\n---\n\n".join(blocks)


context = build_context(results)
print(context[:2500])

[Fuente 1]
Título: La mayoría de los universitarios fallan al distinguir noticias científicas verdaderas y falsas publicadas en redes sociales
Fecha: 23/03/2026
Etiquetas: 
Archivo: 14071444.html
Texto: Título: La mayoría de los universitarios fallan al distinguir noticias científicas verdaderas y falsas publicadas en redes sociales Fecha: 23/03/2026 Etiquetas: Resumen: Una investigación liderada por la Universidad de Oviedo revela que en algunos casos hasta tres cuartas partes del alumnado manifiesta dudas al valorar la veracidad o falsedad de las noticias analizadas | El trabajo concluye que es necesario reforzar la alfabetización científica y mediática para mejorar la capacidad de los estudiantes de identificar información fiable | La muestra se llevó a cabo mediante una encuesta realizada a 221 estudiantes, a los que se pidió evaluar la veracidad de cuatro noticias de temática científica, algunas verdaderas y otras falsas | El análisis indica que los universitarios tienden a basar 

## 7. Generar una respuesta usando el contexto que se le pasa al LLM

Usa un modelo pequeño de Hugging Face si quieres una respuesta más natural.


In [8]:
# Cargar LLM (Llama)
try:
    from transformers import pipeline
    import re
    import torch # Import torch, as device_map might implicitly use it.

    model_id = "meta-llama/Llama-3.2-1B-Instruct"
    pipe = pipeline("text-generation", model=model_id, device_map="auto")
    print(f"Modelo {model_id} cargado.")
except Exception as exc:
    pipe = None
    print(f"No se pudo cargar el modelo: {exc}")


# Función para construir el formato de mensajes
def build_llm_prompt(question: str, retrieved: list[dict]) -> list[dict]:
    context = build_context(retrieved)
    messages = [
        {"role": "system", "content": "Eres un asistente útil que responde preguntas en español basándose exclusivamente en el contexto proporcionado. Devuelve también la fuente si la mencionas en la respuesta. Si no encuentras la respuesta, indícalo."},
        {"role": "user", "content": f"Pregunta:\n{question}\n\nContexto:\n{context}"}
    ]
    return messages


def llm_answer(question: str, retrieved: list[dict], max_new_tokens: int = 250) -> str:
    if pipe is None:
        return "El modelo no está cargado."

    messages = build_llm_prompt(question, retrieved)

    # Use the pipeline directly for generation
    # `return_full_text=False` is crucial to get only the generated part, not the whole prompt.
    outputs = pipe(messages, max_new_tokens=max_new_tokens, do_sample=True, temperature=0.7, return_full_text=False)

    if outputs and outputs[0] and 'generated_text' in outputs[0]:
        answer = outputs[0]['generated_text']
    else:
        answer = "No se pudo generar una respuesta."

    # Clean any specific formatting that might come from the model (e.g., <think> tags from Gemma)
    answer = re.sub(r"<think>.*?</think>", "", answer, flags=re.DOTALL).strip()

    return answer

config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.47GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Modelo meta-llama/Llama-3.2-1B-Instruct cargado.


## 8. Encapsular el RAG completo

Ahora juntamos retrieval y generación en una sola función para probar preguntas rápidamente.


In [9]:
def rag_answer(question: str, top_k: int = 5) -> tuple[str, pd.DataFrame]:
    retrieved = search_bm25(question, top_k=top_k)
    answer = llm_answer(question, retrieved)
    ranking = pd.DataFrame([{
        "rank": r["rank"],
        "score": round(r["score"], 3),
        "fecha": r["fecha"],
        "titulo": r["titulo"],
        "matched_tokens": ", ".join(r["matched_tokens"]),
    } for r in retrieved])
    return answer, ranking


question = "¿Hay investigaciones recientes sobre minería o patrimonio histórico?"
answer, ranking = rag_answer(question, top_k=6)

display(ranking)
print(answer)


[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'temperature', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=250) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


,rank,score,fecha,titulo,matched_tokens
0,1,10.202,20/03/2026,Investigadores de la Universidad de Oviedo rev...,"investigaciones, recientes"
1,2,9.771,06/07/2026,La Universidad de Oviedo reúne a especialistas...,"historico, patrimonio"
2,3,9.543,19/06/2026,Premio de la Comisión Histórica de Cambridge a...,"historico, patrimonio"
3,4,8.916,19/06/2026,Astrónomos de Light Bridges y la Universidad d...,"investigaciones, mineria"
4,5,8.258,12/01/2026,La combinación de un clima extremo y vegetació...,"historico, investigaciones"
5,6,8.231,06/07/2026,La Universidad de Oviedo reúne a especialistas...,"historico, patrimonio"


No hay investigaciones recientes sobre minería o patrimonio histórico que se mencionen en los tres textos proporcionados. Sin embargo, puedo ofrecerte información sobre algunas de las investigaciones y proyectos relacionados con el patrimonio histórico y la minería en España.

En España, existen varios proyectos y investigaciones que se centran en el patrimonio histórico y la conservación de infraestructuras y edificios históricos. Algunos ejemplos son:

* La "Iniciativa Española para la Conservación de los Patrimonios Históricos y Artísticos" (ICHEP), una iniciativa del Ministerio de Cultura y Deportes que busca promover la conservación y la restauración de los patrimonios históricos y artísticos de España.
* El "Programa de Conservación de la Patrimonialidad Histórica y Artística" (PCHA), un proyecto del Ministerio de Cultura y Deportes que busca promover la conservación y la restauración de los patrimonios históricos y artísticos de España.
* La "Red de Protección Histórica y Cultur

## 9. Prueba con tus propias preguntas

Cambia la variable `my_question` y ejecuta la celda. Si quieres probar el modelo local, cambia `use_llm` a `True`.


In [10]:
my_question = "¿Hay noticias sobre cursos de verano o actividades culturales?"

answer, ranking = rag_answer(my_question, top_k=6)
display(ranking)
print(answer)


[transformers] Both `max_new_tokens` (=250) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


,rank,score,fecha,titulo,matched_tokens
0,1,19.211,24/06/2026,La Universidad de Oviedo extiende su oferta ac...,"actividades, culturales, cursos, verano"
1,2,14.496,23/01/2026,El proyecto de dinamización universitaria de l...,"actividades, cursos, verano"
2,3,14.241,24/06/2026,La Universidad de Oviedo extiende su oferta ac...,"culturales, cursos, verano"
3,4,13.346,24/06/2026,La Universidad de Oviedo extiende su oferta ac...,"cursos, verano"
4,5,13.197,24/06/2026,La Universidad de Oviedo extiende su oferta ac...,"culturales, cursos, verano"
5,6,12.938,06/07/2026,La Universidad de Oviedo recibe este verano a ...,"cursos, verano"


No hay una respuesta directa a la pregunta sobre cursos de verano o actividades culturales, ya que las fuentes proporcionadas ofrecen información sobre la extensión académica y la colaboración con instituciones, más que sobre los cursos específicos. Sin embargo, puedo ofrecerte una respuesta general que refleje la información disponible.

La Universidad de Oviedo extiende su oferta académica al mes de julio con doce cursos de verano, que se desarrollarán en las sedes universitarias de Oviedo y Mieres, así como en los concejos de Langreo y Carreño. La programación combina temas científicos, técnicos, culturales, sociales y patrimoniales y se enfoca en abrir la universidad a públicos diversos de todos los territorios más allá de los límites del curso académico.

La vicerrectora de Extensión Universitaria y Proyección Cultural, Marta Mateo, destaca que "los cursos de verano son un símbolo de la extensión universitaria y una de las expresiones más claras de la vocación de servicio público 

## Ideas para experimentar

- Cambia `chunk_size` y `overlap` en `chunk_text` si quieres más o menos contexto.
- Prueba consultas con palabras distintas a las del texto para ver el límite de BM25.

Regla mental: BM25 decide qué leer; el generador decide cómo contestar con lo leído.
